# Rulebase accuracy on HI-Medium (R1 to R7)

Question: are the rules accurate enough to keep, or should the model replace them?

- Each rule (R1 to R7) and the combined rulebase are scored on HI-Medium.
- A rule alert counts on the day it opens. It is scored against laundering in the next `HORIZON` days for the same account.
- Lift = precision / base rate. The bar is **10 times the base rate**, with at least 20 alerts.
- Graph rules R5 to R7 are built over the whole file, so this backtests the rulebase as the app runs it. It is not a point-in-time test.

**Resources:** the whole file is loaded at once. Use `ml.r5.4xlarge` (128 GB) or larger. The run takes hours, mostly in the single-threaded rule loops. Confirm the budget before running the full file.

**Before you run:** upload the transactions to S3: `aws s3 cp backend/docs/HI-Medium_Trans.csv s3://neu-datasets/ --profile <profile>`.

In [ ]:
%pip install -q "pandas==3.0.6" "numpy==2.5.4" "pyarrow==26.0.0" "boto3"

## 1. Configuration

In [ ]:
from pathlib import Path
import json
import shutil
import sys
import time

import boto3
import numpy as np
import pandas as pd

NB_DIR = Path.cwd()  # run from the notebooks/ folder
sys.path.insert(0, str(NB_DIR))

S3_BUCKET = "neu-datasets"
S3_KEY = "HI-Medium_Trans.csv"
DATA_DIR = NB_DIR / "data"
EVAL_DIR = NB_DIR / "eval" / "hi_medium_rules"
LOCAL_CSV = DATA_DIR / S3_KEY
HORIZON = 2          # label horizon in days (the span is about 16 days)
SAMPLE_EVERY = 1     # 1 = full file; set to 30 for a quick check on about 1/30 of the rows
RUN_NAME = "hi_medium_rules"
UPLOAD_RESULTS = True
for folder in (DATA_DIR, EVAL_DIR):
    folder.mkdir(parents=True, exist_ok=True)

## 2. Download the transactions

In [ ]:
s3 = boto3.client("s3")
if not LOCAL_CSV.exists():
    s3.download_file(S3_BUCKET, S3_KEY, str(LOCAL_CSV))
print(f"{LOCAL_CSV.name}: {LOCAL_CSV.stat().st_size / 2**30:.2f} GiB")

## 3. Load the transactions

With `SAMPLE_EVERY > 1` only every Nth row is kept (header always kept), for a quick check of the code path.

In [ ]:
from triage_lib.alerts import load_csv

t0 = time.time()
if SAMPLE_EVERY == 1:
    df = load_csv(str(LOCAL_CSV))
else:
    sample = DATA_DIR / f"HI-Medium_sample{SAMPLE_EVERY}.csv"
    with LOCAL_CSV.open() as src, sample.open("w") as dst:
        for n, line in enumerate(src, start=1):
            if n == 1 or n % SAMPLE_EVERY == 0:
                dst.write(line)
    df = load_csv(str(sample))
print(f"rows: {len(df):,}  span: {df['ts'].min()} to {df['ts'].max()}  ({time.time() - t0:.0f}s)")

## 4. Build the rule alerts (R1 to R7)

This is the slow step: R2 and R3 are single-threaded loops, and R5 to R7 run over the whole edge set.

In [ ]:
from triage_lib.alerts import build_alerts

t0 = time.time()
alerts = build_alerts(df)
print(f"alerts: {len(alerts):,}  ({(time.time() - t0) / 60:.1f} min)")

## 5. Score each rule and the combined rulebase

In [ ]:
from triage_lib.rules_eval import evaluate_rules

result = evaluate_rules(df, alerts, horizon=HORIZON)
rows = []
for name, m in {**result["rules"], **result["combined"]}.items():
    rows.append({"rule": name, **m})
table = pd.DataFrame(rows).set_index("rule")
display(table[["alerts", "tp", "fp", "precision", "base_rate", "lift", "recall", "alerts_per_day",
               "conclusive", "passes_lift_bar"]])
print("base rate:", result["base_rate"], " population rows:", result["population_rows"],
      " positive rows:", result["positive_rows"], " days:", result["days_evaluated"])

## 6. Verdict

A rule is kept only if it has at least 20 alerts and lift of at least 10. The combined rulebase has to pass too. Anything that fails is listed below.

In [ ]:
failing = [n for n, m in {**result["rules"], **result["combined"]}.items()
           if not m["passes_lift_bar"]]
print("passes the lift bar:", [n for n in table.index if n not in failing])
print("fails or not conclusive:", failing)
print(result["caveat"])

## 7. Save the results

In [ ]:
out = EVAL_DIR / "rules_metrics.json"
out.write_text(json.dumps(result, indent=2), encoding="utf-8")
print("saved", out)

if UPLOAD_RESULTS:
    s3.upload_file(str(out), S3_BUCKET, f"triage/{RUN_NAME}/rules_metrics.json")
    print(f"uploaded s3://{S3_BUCKET}/triage/{RUN_NAME}/rules_metrics.json")